In [ ]:
BANNER = \
"""\
# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [ ]:
# ------------------------------------------------------------
# from analytical dataset to modelling datasets
# ------------------------------------------------------------

# MoP_04 | From Dataset to Model
# notebook 1 | feature representation

# input:
#   01_input/penguins_analytical_dataset.csv
#
# outputs:
#   03_output/penguins_modelling_[A-E]_*.csv
#   03_output/penguins_modelling_[A-E]_*.tab
#   03_output/penguins_modelling_A_study_noOHE.csv
#   03_output/penguins_modelling_A_study_noOHE.tab
#   03_output/penguins_modelling_[A-E]_*.joblib

# goal:
#   construct modelling datasets for different
#   prediction contexts, using appropriate feature representations

#   construct five modelling datasets for different prediction contexts
#   and an alternative representation of scenario A without OHE
#   to also explore the relationship between feature representation
#   and learning algorithms

In [ ]:
# ------------------------------------------------------------
# define input and output paths
# ------------------------------------------------------------

from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "02_code": ROOT = ROOT.parent

INPUT = ROOT / "01_input" / "penguins_analytical_dataset.csv"
OUTPUT = ROOT / "03_output"

assert INPUT.exists(), f"PTS | missing input: {INPUT}"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("input:", INPUT)
print("output:", OUTPUT)

In [ ]:
# ------------------------------------------------------------
# load and inspect the analytical dataset
# ------------------------------------------------------------

import pandas as pd

analytical = pd.read_csv( INPUT )

print( "shape:", analytical.shape )
print( "columns:", analytical.columns.tolist() )

display( analytical.head() )

In [ ]:
# ------------------------------------------------------------
# validate the analytical dataset
# ------------------------------------------------------------

expected_columns = {
  "island", "year", "sex",
  "bill_length_mm", "bill_depth_mm",
  "flipper_length_mm", "body_mass_g",
  "species"
}

assert len(analytical) == 333, "PTS | unexpected number of observations"
assert set(analytical.columns) == expected_columns, "PTS | unexpected attributes"
assert not analytical.isna().any().any(), "PTS | missing values found"

print("analytical dataset: valid")

In [ ]:
# ------------------------------------------------------------
# define prediction scenarios
# ------------------------------------------------------------

# A — historical field study
# classify penguins within the original field-study setting
# where all recorded information is available

# B — anywhere in Antarctica
# classify penguins observed anywhere in Antarctica
# without requiring information about their observation location

# C — whenever needed
# classify penguins observed at the present time
# without relying on the historical period of data collection

# D — biological profile
# classify penguins independently of geographic and temporal
# context, using available biological information

# E — morphological measurements
# classify penguins using only body measurements
# without requiring prior knowledge of biological or field context

In [ ]:
# ------------------------------------------------------------
# define scenario attributes
# ------------------------------------------------------------

# TODO | select the morphological attributes (common for all scenarios A-D
morphological = [
  ...
]

# TODO | select the additional attributes for scenarios A-D
scenarios = {
  "A": ... + morphological,
  "B": ... + morphological,
  "C": ... + morphological,
  "D": ... + morphological,
  "E": morphological
}

for scenario, attributes in scenarios.items():
  print( f"{scenario}: {attributes}" )

In [ ]:
# ------------------------------------------------------------
# define attribute types and nominal categories
# ------------------------------------------------------------

# numerical attributes
numeric = [
  "year", "bill_length_mm", "bill_depth_mm",
  "flipper_length_mm", "body_mass_g"
]

# nominal attributes and their categories
categorical_nominal = {
  "island": ["Biscoe", "Dream", "Torgersen"],
  "sex": ["FEMALE", "MALE"]
}

In [ ]:
# ------------------------------------------------------------
# transform selected attributes
# ------------------------------------------------------------

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def transform_attributes(data, attributes):
  attr_num = [ a for a in attributes if a in numeric ]
  attr_cat = [ a for a in attributes if a in categorical_nominal ]
  categories = [ categorical_nominal[ attr ] for attr in attr_cat ]

  # numerical attributes: preserve original values (i.e., "passthrough")
  # nominal attributes: apply one-hot encoding with predefined categories and sparse_output=False

  # TODO | configure numerical and nominal transformations
  transformer_num = ( "numeric", ..., attr_num )
  transformer_cat = ( "nominal", ..., attr_cat )

  # TODO | indicate the transformations to apply
  transformer = ColumnTransformer(
    transformers=[ ... ],
    verbose_feature_names_out=False )

  transformer.fit( data[ attributes ] )
  values = transformer.transform( data[ attributes ] )

  features = pd.DataFrame(
    data=values,
    columns=transformer.get_feature_names_out(),
    index=data.index )

  return features, transformer

In [ ]:
# ------------------------------------------------------------
# test attribute transformation
# ------------------------------------------------------------

features, transformer = transform_attributes( analytical, scenarios[ "A" ] )

print( "shape:", features.shape )
display( features.head() )

# expected output (scenario A):
# shape: (333, 10)
#
# first row (index 0):
# | year=2007.0 | bill_length_mm=39.5 | bill_depth_mm=17.8 |
# | flipper_length_mm=188.0 | body_mass_g=3300.0 |
# | island_Biscoe=0.0 | island_Dream=1.0 | island_Torgersen=0.0 |
# | sex_FEMALE=1.0 | sex_MALE=0.0

In [ ]:
# ------------------------------------------------------------
# construct and save modelling datasets and transformers
# ------------------------------------------------------------

import joblib

# output filenames for each prediction scenario
filenames = {
  "A": "penguins_modelling_A_study.csv",
  "B": "penguins_modelling_B_anywhere.csv",
  "C": "penguins_modelling_C_whenever.csv",
  "D": "penguins_modelling_D_biological.csv",
  "E": "penguins_modelling_E_morphological.csv"
}

# important:
# the construction of each modelling dataset includes an additional
# technical attribute, ID, to identify the same observation across all scenarios
# - this enables traceability and comparison of predictions from different models
# - id is not an explanatory attribute and must not be used as a model feature

# materialize the modelling dataset and preserve the fitted transformer for each scenario

# TODO | identify the scenario and its attributes
for ... in scenarios.items():
  # transform the selected attributes into model features and keep transformer
  features, transformer = transform_attributes( analytical, attributes )

  # assign a common identifier based on the original observation order
  modelling_dataset = features.copy()

  # all scenarios are derived from the same analytical dataset
  # feature transformations preserve the number and order of observations
  # sequential IDs therefore identify the same observation across scenarios
  # ID supports traceability and comparison, but is not a model feature
  modelling_dataset.insert( 0, "ID", range( 1, len( modelling_dataset ) + 1 ) )

  # preserve the original target values
  modelling_dataset[ "species" ] = analytical[ "species" ]

  # materialize the modelling dataset
  filename = OUTPUT / filenames[ scenario ]
  modelling_dataset.to_csv( filename, index=False )

  # save the fitted transformer for this scenario
  transformer_path = filename.with_suffix( ".joblib" )
  joblib.dump( transformer, transformer_path )

  print( f"{scenario}: {modelling_dataset.shape}\t -> {filename.name}" )
  print( "> transformer:\t ->", transformer_path.name )
  print( "  - name:\t", type( transformer ).__name__ )
  print( "  - input-attributes:", transformer.feature_names_in_.tolist() )
  print( "  - output-features:", transformer.get_feature_names_out().tolist() )
  print()

In [ ]:
# ------------------------------------------------------------
# construct scenario A without one-hot encoding
# ------------------------------------------------------------

# preserve the original representation of nominal attributes

# copy the analytical dataset
scenario_A_noOHE = "A"

# TODO | select scenario A attributes without transformation
modelling_dataset_A_noOHE = analytical[ ... ].copy()

# add the same technical identifier used in the other datasets
# TODO | generate sequential identifiers starting at 1
modelling_dataset_A_noOHE.insert( 0, "ID", range( ... ) )

# TODO | preserve the original target values
modelling_dataset_A_noOHE[ "species" ] = analytical[ ... ]

# save the alternative representation
filename = OUTPUT / "penguins_modelling_A_study_noOHE.csv"
modelling_dataset_A_noOHE.to_csv( filename, index=False )

print( f"{scenario_A_noOHE} (noOHE): {modelling_dataset_A_noOHE.shape}\t -> {filename.name}" )

In [ ]:
# ------------------------------------------------------------
# inspect modelling datasets
# ------------------------------------------------------------

for scenario, filename in filenames.items():
  modelling_dataset = pd.read_csv( OUTPUT / filename )

  # exclude the technical identifier ("ID") and target ("species")
  features = modelling_dataset.drop( columns=[ "ID", "species" ] )

  print( f"\n{scenario} — {filename}" )
  print( f"observations:\t {len( modelling_dataset )}" )
  print( f"features:\t {features.shape[1]}" )
  print( f"feature names:\t {features.columns.tolist()}" )

# inspect the alternative representation of scenario A
# exclude the technical identifier ("ID") and target ("species")
features = modelling_dataset_A_noOHE.drop( columns=[ "ID", "species" ] )

print( "\nA (no OHE) — penguins_modelling_A_study_noOHE.csv" )
print( f"observations:\t {len( modelling_dataset_A_noOHE )}" )
print( f"features:\t {features.shape[1]}" )
print( f"feature names:\t {features.columns.tolist()}" )

In [ ]:
# ------------------------------------------------------------
# validate modelling dataset features
# ------------------------------------------------------------

# important: this checkpoint validates feature names in all scenarios
# and feature order in scenarios with fitted transformers
# it does not verify transformed values or observation integrity

import hashlib

# reference (expected) hashes generated from the solution
expected_hashes = {
  'A': 'f67eb725a1dd91481b2a11c04f1901bfd63b1a8ca96e32409941c14c5bfe9f3b',
  'B': 'a7680882199ee5bc7b7241bfbe085469499f713bcfc9e08fe5b2e12b8ee88900',
  'C': '4e98f56d2cd9060633110d233f600cb1dfb7260c85c49df739d992ffee670f6c',
  'D': '524fd4453bcd2f0e7b2a5bc226be8b7edfdfab7da1e9ff78e77508a6b096a972',
  'E': 'cc78ce84ed9613cf7173dba0cb6723ecb330511ad000df8d87a502486f060041',
  'A_noOHE': 'de43b5b6f1543bf212646a12a6b292724adf97b550ed0601f012020b41102f9e'}

# compute an order-independent hash of feature names
def feature_hash( features ):
  names = "|".join( sorted( features ) )
  return hashlib.sha256( names.encode( "utf-8" ) ).hexdigest()


# include the alternative representation of scenario A
validation_files = filenames.copy()
validation_files[ "A_noOHE" ] = "penguins_modelling_A_study_noOHE.csv"

# validate the features of each materialized dataset
valid = True
for scenario, filename in validation_files.items():
  modelling_dataset = pd.read_csv( OUTPUT / filename )
  features = modelling_dataset.drop( columns=[ "ID", "species" ] )

  # validate feature names and order against the fitted transformer
  if scenario != "A_noOHE":
    transformer = joblib.load( ( OUTPUT / filename ).with_suffix( ".joblib" ) )
    expected_features = transformer.get_feature_names_out().tolist()
    assert features.columns.tolist() == expected_features, f"PTS | invalid feature order in scenario {scenario}"

  # compare the actual hash with the reference (expected) hash
  valid_scenario = feature_hash( features.columns ) == expected_hashes[ scenario ]
  print( f"{scenario}: {'PASS' if valid_scenario else 'FAIL'}" )
    
  valid = valid and valid_scenario

assert valid, "PTS | invalid feature selection in one or more scenarios"
print( "modelling dataset features in all scenarios: valid" )

In [ ]:
# ------------------------------------------------------------
# export modelling datasets for Orange DM
# ------------------------------------------------------------

# Orange .tab files use tab separators and three header rows:
#   row 1: attribute names
#   row 2: data types (c = continuous, d = discrete)
#   row 3: roles (class = target, meta = additional information)
#
# reference (Orange 3 documentation):
# https://orange3.readthedocs.io/en/latest/reference/data.io.html
#
# ID is a meta attribute, not a model feature
# species is the target; all remaining columns are numerical features

for scenario, filename in filenames.items():
  modelling_dataset = pd.read_csv( OUTPUT / filename )
  tab_file = ( OUTPUT / filename ).with_suffix( ".tab" )

  # define data types and roles
  types = []
  roles = []

  for col in modelling_dataset.columns:
    if col == "species":
      types.append( "d" )
      roles.append( "class" )
    elif col == "ID":
      types.append( "c" )
      roles.append( "meta" )
    else:
      types.append( "c" )
      roles.append( "" )

  # prepend metadata rows to the observations
  metadata = pd.DataFrame(
    data=[ types, roles ],
    columns=modelling_dataset.columns )

  orange_dataset = pd.concat(
    [ metadata, modelling_dataset ],
    ignore_index=True )

  # export column names, metadata and observations
  orange_dataset.to_csv( tab_file, sep="\t", index=False )

  print( f"{scenario}: {tab_file.name}" )

In [ ]:
# ------------------------------------------------------------
# export scenario A without one-hot encoding for Orange DM
# ------------------------------------------------------------

# preserve island and sex as discrete attributes
# species is the target; ID is a meta attribute

tab_file = OUTPUT / "penguins_modelling_A_study_noOHE.tab"

# define data types and roles
types = []
roles = []

for col in modelling_dataset_A_noOHE.columns:
  if col == "species":
    types.append( "d" )
    roles.append( "class" )
  elif col == "ID":
    types.append( "c" )
    roles.append( "meta" )
  elif col in categorical_nominal:
    # TODO | indicate the data type and role of nominal attributes
    types.append( ... )
    roles.append( ... )
  else:
    types.append( "c" )
    roles.append( "" )

# prepend metadata rows to the observations
# TODO | construct the metadata DataFrame
metadata = pd.DataFrame( ... )

# TODO | construct the Orange dataset with metadata and observations
orange_dataset = pd.concat( ... )

# export column names, metadata and observations
orange_dataset.to_csv( tab_file, sep="\t", index=False )

print( f"A (no OHE): {tab_file.name}" )

In [ ]:
# ------------------------------------------------------------
# validate scenario A without one-hot encoding
# ------------------------------------------------------------

# reconstruct the expected dataset
expected = analytical[ scenarios[ "A" ] ].copy()
expected.insert( 0, "ID", range( 1, len( expected ) + 1 ) )
expected[ "species" ] = analytical[ "species" ]

# validate the CSV dataset
actual = pd.read_csv( OUTPUT / "penguins_modelling_A_study_noOHE.csv" )
assert actual.equals( expected ), "PTS | invalid no-OHE CSV dataset"

# validate the TAB observations
tab_data = pd.read_csv(
  OUTPUT / "penguins_modelling_A_study_noOHE.tab", sep="\t", skiprows=[ 1, 2 ] )
assert tab_data.equals( actual ), "PTS | invalid no-OHE TAB observations"

print( "scenario A (no OHE): PASS" )

In [ ]:
# ------------------------------------------------------------
# final checkpoint — modelling datasets
# ------------------------------------------------------------

# five prediction scenarios and six feature representations
# - are ready for predictive learning
# scenario A includes an alternative representation without OHE
#
# fitted feature transformers have been saved for scenarios A–E
# they can be reused to process new observations during deployment
#
# next step: train, predict and evaluate models
# comparing learners, prediction contexts and feature representations

print( "checkpoint: six modelling datasets generated; feature names validated" )
print( "feature transformers: five saved for deployment" )
print( "output:", OUTPUT )